## PRACTICA OBLIGATORIA: **Ensembles: Bagging y Boosting**

* La práctica consiste en obtener el mejor modelo para un problema de clasificación sobre diabetes en mujeres de ascendencia india en EEUU.
* Recuerda subir el notebook a tu repositorio antes de la sesión en vivo.
* No es necesario que esté perfecta, sólo que se vea el esfuerzo.


### Ejercicio 0

Importa los paquetes y módulos que necesites a lo largo del notebook.

In [ ]:
import bootcampviztools as bt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV, train_test_split


### Descripción del dataset

El dataset de los Pima Indians Diabetes contiene datos de mujeres de al menos 21 años de ascendencia india Pima (Phoenix, Arizona). El objetivo es predecir si desarrollará diabetes en los próximos cinco años.

| Variable | Descripción |
|---|---|
| `preg` | Número de embarazos |
| `plas` | Concentración de glucosa en plasma (2h) |
| `press` | Presión arterial diastólica (mm Hg) |
| `skin` | Grosor del pliegue cutáneo del tríceps (mm) |
| `test` | Insulina en suero a 2 horas (mu U/ml) |
| `mass` | IMC — peso(kg) / altura(m)² |
| `pedi` | Función del pedigree de diabetes (predisposición genética) |
| `age` | Edad en años |
| `class` | **Target**: 1 = desarrolló diabetes, 0 = no |


### Carga de datos

In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
names = ['preg', 'plas', 'press', 'skin', 'test', 'mass', 'pedi', 'age', 'class']

df = pd.read_csv(url, header=None, names=names)
df.head()


In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
df[["preg", "age"]].hist()
plt.tight_layout()

In [ ]:
df.preg.value_counts()

*`preg` puede ser numérica o categórica, la analizaremos de las dos formas contra el target. De momento, todas numéricas.*

In [ ]:
target = "class"
features_num = [c for c in df.columns if c != target]
features_num

#### Distribución del target

In [ ]:
bt.pinta_distribucion_categoricas(df, [target], mostrar_valores=True, relativa=True)

*Ligeramente desbalanceado. No haremos resampling, pero usaremos `class_weight='balanced'` / `scale_pos_weight` en los modelos.*

## Train-Test

In [ ]:
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42, stratify=df[target])

print("Distribución en train:")
print(train_set[target].value_counts(normalize=True).round(3))
print("\nDistribución en test:")
print(test_set[target].value_counts(normalize=True).round(3))

In [ ]:
train_set.describe()

*Los ceros en `plas`, `press`, `skin`, `test`, `mass` no son mediciones reales — son valores faltantes codificados como 0. Los tratamos como NaN.*

In [ ]:
tienen_nulos = ["plas", "press", "skin", "test", "mass"]

train_set[tienen_nulos] = train_set[tienen_nulos].replace(0, np.nan)
test_set[tienen_nulos] = test_set[tienen_nulos].replace(0, np.nan)

In [ ]:
from sklearn.impute import SimpleImputer

# fit solo sobre train, transform sobre ambos -> evita data leakage
imp = SimpleImputer(strategy="median")
train_set[tienen_nulos] = imp.fit_transform(train_set[tienen_nulos])
test_set[tienen_nulos] = imp.transform(test_set[tienen_nulos])

train_set[tienen_nulos].isna().sum()

In [ ]:
bt.plot_categorical_relationship(train_set, target, "preg", relative_freq=True, show_values=True)

In [ ]:
bt.plot_grouped_histograms(train_set, cat_col=target, num_col="preg", group_size=3)

*No vemos motivos para cambiar `preg` a categórica. Continuamos con todas las variables numéricas.*

In [ ]:
for col in features_num:
    bt.plot_grouped_histograms(train_set, cat_col=target, num_col=col, group_size=3)

*plas, mass y age parecen las más predictoras — lo confirmaremos con el feature importance del RandomForest.*

*Como vamos a usar árboles en todos los modelos, no es necesario transformar ni escalar las variables.*

In [ ]:
X_train = train_set[features_num].copy()
y_train = train_set[target]

X_test = test_set[features_num].copy()
y_test = test_set[target]

### Baseline: RandomForest con parámetros por defecto

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

base_clf = RandomForestClassifier(max_depth=5, random_state=42)

In [ ]:
scores = cross_val_score(base_clf, X_train, y_train, cv=5, scoring="balanced_accuracy")
print(f"Balanced Accuracy medio (CV-5): {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
base_clf.fit(X_train, y_train)
print("--- RandomForest baseline (train) ---")
print(classification_report(y_train, base_clf.predict(X_train)))

In [ ]:
df_fi = pd.DataFrame({
    "feature": base_clf.feature_names_in_,
    "importance": base_clf.feature_importances_
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=df_fi, x="importance", y="feature", palette="viridis", ax=ax)
ax.set_title("Feature Importance — RandomForest baseline")
plt.tight_layout()
plt.show()

print(df_fi.to_string(index=False))


*plas, mass y age son las más predictoras. Confirma nuestras hipótesis del EDA, aunque preg tiene menos importancia de la esperada.*

### Comparación baseline de los tres modelos (sin optimizar)

In [ ]:
xgb_clf = XGBClassifier(max_depth=5, random_state=42)
lgb_clf = LGBMClassifier(max_depth=5, random_state=42, verbose=-1, n_jobs=-1)

for nombre, modelo in zip(
    ["Random Forest", "XGBoost", "LightGBM"],
    [base_clf, xgb_clf, lgb_clf]
):
    scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="balanced_accuracy")
    print(f"{nombre:15s} | Balanced Accuracy CV-5: {scores.mean():.4f} ± {scores.std():.4f}")


*Seleccionamos el mejor como punto de partida. Si el tiempo lo permite, lo ideal es optimizar los tres y comparar sus mejores versiones.*

In [ ]:
# Ratio de clases para scale_pos_weight de XGBoost (equivalente a class_weight='balanced')
ratio_clases = len(train_set[train_set[target] == 0]) / len(train_set[train_set[target] == 1])
print(f"scale_pos_weight = {ratio_clases:.4f}  (negativos/positivos en train)")


### Optimización de hiperparámetros

#### XGBoost

In [ ]:
param_grid_xgb = {
    "n_estimators": [100, 200, 400],
    "eta": [0.1, 0.3, 0.6, 1],
    "max_depth": [1, 6, 10, 20],
    "min_child_weight": [1, 10, 20, 100],
    "scale_pos_weight": [ratio_clases, 1],
    "colsample_bytree": [0.5, 1]
}

xgb_clf = XGBClassifier(random_state=42)

xgb_grid = GridSearchCV(xgb_clf,
                        param_grid=param_grid_xgb,
                        cv=5,
                        scoring="balanced_accuracy",
                        n_jobs=-1)

xgb_grid.fit(X_train, y_train)


In [ ]:
print('Mejores parámetros XGBoost:', xgb_grid.best_params_)

In [ ]:
print(f'Mejor balanced_accuracy CV-5: {xgb_grid.best_score_:.4f}')

In [ ]:
print("--- XGBoost optimizado (test) ---")
print(classification_report(y_test, xgb_grid.best_estimator_.predict(X_test)))

#### RandomForest

In [ ]:
param_grid_rf = {
    "n_estimators": [100, 200, 400],
    "max_depth": [1, 5, 10, None],
    "min_samples_leaf": [1, 10, 20, 100],
    "class_weight": ["balanced", None],
    "max_features": ["sqrt", "log2", None]
}

rf_clf = RandomForestClassifier(random_state=42)

rf_grid = GridSearchCV(rf_clf,
                       param_grid=param_grid_rf,
                       cv=5,
                       scoring="balanced_accuracy",
                       n_jobs=-1)

rf_grid.fit(X_train, y_train)


In [ ]:
print('Mejores parámetros RF:', rf_grid.best_params_)

In [ ]:
print(f'Mejor balanced_accuracy CV-5: {rf_grid.best_score_:.4f}')

In [ ]:
print("--- RandomForest optimizado (test) ---")
print(classification_report(y_test, rf_grid.best_estimator_.predict(X_test)))


#### LightGBM

In [ ]:
param_dist_lgb = {
    "n_estimators": [100, 200, 400],
    "learning_rate": [0.01, 0.05, 0.1, 0.3],
    "max_depth": [3, 6, 10, -1],
    "num_leaves": [15, 31, 63, 127],
    "min_data_in_leaf": [1, 10, 20, 100],
    "class_weight": ["balanced", None],
    "max_bin": [40, 80, 200]
}

lgb_clf = LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1)

lgb_grid = RandomizedSearchCV(lgb_clf,
                               param_distributions=param_dist_lgb,
                               n_iter=60,
                               cv=5,
                               scoring="balanced_accuracy",
                               random_state=42,
                               n_jobs=-1)

lgb_grid.fit(X_train, y_train)


In [ ]:
print('Mejores parámetros LightGBM:', lgb_grid.best_params_)

In [ ]:
print(f'Mejor balanced_accuracy CV-5: {lgb_grid.best_score_:.4f}')

In [ ]:
print("--- LightGBM optimizado (test) ---")
print(classification_report(y_test, lgb_grid.best_estimator_.predict(X_test)))


### Comparación final de los tres modelos optimizados

In [ ]:
print("=== Balanced Accuracy en CV (sobre train) ===")
for nombre, grid in zip(["XGBoost", "RandomForest", "LightGBM"],
                        [xgb_grid, rf_grid, lgb_grid]):
    print(f"  {nombre:15s}: {grid.best_score_:.4f}")

print("\n=== Classification Report en test ===")
for nombre, grid in zip(["XGBoost", "RandomForest", "LightGBM"],
                        [xgb_grid, rf_grid, lgb_grid]):
    print(f"\n--- {nombre} ---")
    print(classification_report(y_test, grid.best_estimator_.predict(X_test)))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
modelos = [xgb_grid.best_estimator_, rf_grid.best_estimator_, lgb_grid.best_estimator_]
nombres = ["XGBoost", "RandomForest", "LightGBM"]

for ax, modelo, nombre in zip(axes, modelos, nombres):
    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test, ax=ax, colorbar=False)
    ax.set_title(nombre)

plt.suptitle("Matrices de confusión — modelos optimizados", y=1.02)
plt.tight_layout()
plt.show()


*Elegimos el modelo con mejor equilibrio entre balanced_accuracy en CV y métricas en test.
En un problema médico como este, el **recall de la clase 1** (detectar diabéticos reales)
suele ser la métrica más importante: un falso negativo tiene consecuencias mucho más graves que un falso positivo.*
